In [12]:
from functools import partial
from typing import Any, Dict, Type, Union

import torch
import torch.nn as nn

from torch_pointcloud.layers.activations import HardMish, HardSigmoid, HardSwish, Mish, QuickGELU, Sigmoid, Swish, Tanh

In [14]:
# From https://github.com/huggingface/pytorch-image-models/blob/main/timm/layers/create_act.py
# PyTorch has an optimized, native 'silu' (aka 'swish') operator as of PyTorch 1.7.
# Also hardsigmoid, hardswish, and soon mish. This code will use native version if present.
# Eventually, the custom SiLU, Mish, Hard*, layers will be removed and only native variants will be used.
_has_silu = "silu" in dir(torch.nn.functional)
_has_hardswish = "hardswish" in dir(torch.nn.functional)
_has_hardsigmoid = "hardsigmoid" in dir(torch.nn.functional)
_has_mish = "mish" in dir(torch.nn.functional)

_ACT_LAYERS: Dict[str, Type[nn.Module]] = dict(
    silu=nn.SiLU if _has_silu else Swish,
    swish=nn.SiLU if _has_silu else Swish,
    mish=nn.Mish if _has_mish else Mish,
    relu=nn.ReLU,
    relu6=nn.ReLU6,
    leaky_relu=nn.LeakyReLU,
    elu=nn.ELU,
    prelu=nn.PReLU,
    celu=nn.CELU,
    selu=nn.SELU,
    gelu=nn.GELU,
    gelu_tanh=partial(nn.GELU, approximate="tanh"),
    quick_gelu=QuickGELU,
    sigmoid=Sigmoid,
    tanh=Tanh,
    hard_sigmoid=nn.Hardsigmoid if _has_hardsigmoid else HardSigmoid,
    hard_swish=nn.Hardswish if _has_hardswish else HardSwish,
    hard_mish=HardMish,
    identity=nn.Identity,
)


def get_act(name: Union[Type[nn.Module], nn.Module, partial, str], **kwargs: Any) -> nn.Module:
    if isinstance(name, nn.Module):
        return name
    elif isinstance(name, partial):
        return name(**kwargs)
    elif isinstance(name, str):
        return _ACT_LAYERS[name](**kwargs)
    
    print("HERE")
    return name(**kwargs)

In [15]:
get_act(Tanh(), inplace=True)

Tanh()

In [16]:
act_layer = partial(nn.GELU, approximate="tanh")
get_act(act_layer)

GELU(approximate='tanh')

In [21]:
from torch_geometric.nn import MLP


mlp = MLP(channel_list=[10, 20, 50], dropout=0.2, plain_last=False)

print(f"{mlp.lins = }")
print(f"{mlp.norms = }")
print(f"{mlp.dropout = }")
print(f"{mlp.act = }")

mlp.lins = ModuleList(
  (0): Linear(10, 20, bias=True)
  (1): Linear(20, 50, bias=True)
)
mlp.norms = ModuleList(
  (0): BatchNorm(20)
  (1): BatchNorm(50)
)
mlp.dropout = [0.2, 0.2]
mlp.act = ReLU()


In [20]:
from functools import partial

partial(MLP, [10, 20, 50], dropout=0.2)().channel_list

[10, 20, 50]

In [74]:
from typing import List, Optional, Union

import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

from torch_pointcloud.layers.activations import ACT_TYPE, create_act
from torch_pointcloud.layers.norms import NORM_TYPE, create_norm


class MLP(nn.Module):
    def __init__(
        self,
        dims: List[int],
        *,
        act_layer: Optional[Union[ACT_TYPE, List[ACT_TYPE]]] = "relu",
        act_first: bool = False,
        norm_layer: Optional[Union[NORM_TYPE, List[NORM_TYPE]]] = "batch_norm1d",
        dropout: Optional[Union[float, List[float]]] = None,
        bias: Union[bool, List[bool]] = True,
        plain_last: bool = True,
    ) -> None:
        super().__init__()

        N = len(dims)
        N1 = N - 1
        N2 = N - 2 if plain_last else N1
        if N < 2:
            raise ValueError(f"The MLP must have at least 2 dimensions. Got {N}.")

        # Format activations (one for each layer except the last one)
        if act_layer is None:
            act_layers: List[nn.Module] = [nn.Identity()] * N2
        elif isinstance(act_layer, list):
            act_layers = [create_act(act) for act in act_layer]
        else:
            act_layers = [create_act(act_layer)] * N2

        # Format batch norms (one for each layer except the last one)
        if norm_layer is None:
            norm_layers: List[nn.Module] = [nn.Identity()] * N2
        else:
            norm_layer = norm_layer if isinstance(norm_layer, list) else [norm_layer] * N2
            norm_layers = [create_norm(norm, dim) for norm, dim in zip(norm_layer, dims[1:])]

        # Format dropouts (one for each layer except the last one)
        if dropout is None:
            dropouts = [0.0] * N2
        elif isinstance(dropout, list):
            dropouts = dropout
        else:
            dropouts = [dropout] * N2

        # Format biases (one for each layer)
        if isinstance(bias, list):
            biases = bias
        else:
            biases = [bias] * N1

        # Sanity check
        if len(act_layers) != N2 or len(norm_layers) != N2 or len(dropouts) != N2 or len(biases) != N1:
            raise ValueError(
                "The number of activation layers, batch norm layers, and dropouts must be equal to the number of layers (-1 if plain_last is `True`) in the MLP. "
                f"Got {len(act_layers)}, {len(norm_layers)}, {len(dropouts)}, and {N1} (with {plain_last=}) respectively."
            )

        lins = [nn.Linear(in_dim, out_dim, bias=bias) for in_dim, out_dim, bias in zip(dims[:-1], dims[1:], biases)]
        self.linear_layers = nn.ModuleList(lins)
        self.act_layers = nn.ModuleList(act_layers)
        self.norm_layers = nn.ModuleList(norm_layers)
        self.dropouts = dropouts
        self.act_first = act_first
        self.plain_last = plain_last

    def forward(self, x: Tensor) -> Tensor:
        # If `plain_last=True`, then `len(norm_layers) = len(act_layers) = len(dropouts) = len(linear_layers) - 1,
        # thus skipping the execution of the last layer inside the for-loop.
        for lin, norm, act, dropout in zip(self.linear_layers, self.norm_layers, self.act_layers, self.dropouts):
            x = lin(x)
            if self.act_first:
                x = act(x)
            x = norm(x)
            if not self.act_first:
                x = act(x)
            x = F.dropout(x, p=dropout, training=self.training)

        # If `plain_last=True`, then the last layer is executed here.
        if self.plain_last:
            x = self.linear_layers[-1](x)

        return x
    
    # def __repr__(self) -> str:
    #     dims = [lin.in_features for lin in self.linear_layers] + [self.linear_layers[-1].out_features]
    #     extra_repr = f"dims={dims}, act_first={self.act_first}, plain_last={self.plain_last}"
    #     return f"{self.__class__.__name__}({extra_repr})"

In [76]:
mlp = MLP([10, 20, 50, 100], norm_layer=None, dropout=None)
mlp

MLP(
  (linear_layers): ModuleList(
    (0): Linear(in_features=10, out_features=20, bias=True)
    (1): Linear(in_features=20, out_features=50, bias=True)
    (2): Linear(in_features=50, out_features=100, bias=True)
  )
  (act_layers): ModuleList(
    (0-1): 2 x ReLU()
  )
  (norm_layers): ModuleList(
    (0-1): 2 x Identity()
  )
)

In [66]:
print(f"{mlp.linear_layers = }")
print(f"{mlp.act_layers = }")
print(f"{mlp.norm_layers = }")
print(f"{mlp.dropouts = }")
print(f"{mlp.act_first = }")

mlp.linear_layers = ModuleList(
  (0): Linear(in_features=10, out_features=20, bias=True)
  (1): Linear(in_features=20, out_features=50, bias=True)
  (2): Linear(in_features=50, out_features=100, bias=True)
)
mlp.act_layers = ModuleList(
  (0-1): 2 x Identity()
)
mlp.norm_layers = ModuleList(
  (0-1): 2 x Identity()
)
mlp.dropouts = [0.0, 0.0]
mlp.act_first = False


In [57]:
x = torch.randn(32, 10)
out = mlp(x)

In [79]:
class SharedMLP(nn.Module):
    def __init__(
        self,
        channels: List[int],
        *,
        act_first: bool = False,
        act_layer: Optional[Union[ACT_TYPE, List[ACT_TYPE]]] = "relu",
        norm_layer: Optional[Union[NORM_TYPE, List[NORM_TYPE]]] = "batch_norm1d",
        conv_layer: Type[nn.Module] = nn.Conv1d,
        dropout: Optional[Union[float, List[float]]] = None,
        bias: Union[bool, List[bool]] = True,
        plain_last: bool = True,
    ):
        super().__init__()
        N = len(channels)
        N1 = N - 1
        N2 = N - 2 if plain_last else N1
        if N < 2:
            raise ValueError(f"The SharedMLP must have at least 2 channels. Got {N}.")

        # Format activations (one for each layer except the last one)
        if act_layer is None:
            act_layers: List[nn.Module] = [nn.Identity()] * N2
        elif isinstance(act_layer, list):
            act_layers = [create_act(act) for act in act_layer]
        else:
            act_layers = [create_act(act_layer)] * N2

        # Format batch norms (one for each layer except the last one)
        if norm_layer is None:
            norm_layers: List[nn.Module] = [nn.Identity()] * N2
        else:
            norm_layer = norm_layer if isinstance(norm_layer, list) else [norm_layer] * N2
            norm_layers = [create_norm(norm, channel) for norm, channel in zip(norm_layer, channels[1:])]

        # Format dropouts (one for each layer except the last one)
        if dropout is None:
            dropouts = [0.0] * N2
        elif isinstance(dropout, list):
            dropouts = dropout
        else:
            dropouts = [dropout] * N2

        # Format biases (one for each layer)
        if isinstance(bias, list):
            biases = bias
        else:
            biases = [bias] * N1

        # Sanity check
        if len(act_layers) != N2 or len(norm_layers) != N2 or len(dropouts) != N2 or len(biases) != N1:
            raise ValueError(
                "The number of activation layers, batch norm layers, and dropouts must be equal to the number of layers (-1 if plain_last is `True`) in the SharedMLP. "
                f"Got {len(act_layers)}, {len(norm_layers)}, {len(dropouts)}, and {N1} (with {plain_last=}) respectively."
            )

        convs = [
            conv_layer(in_channels, out_channels, bias=bias, kernel_size=1, stride=1)
            for in_channels, out_channels, bias in zip(channels[:-1], channels[1:], biases)
        ]
        self.convs = nn.ModuleList(convs)
        self.acts = nn.ModuleList(act_layers)
        self.norms = nn.ModuleList(norm_layers)
        self.dropouts = dropouts
        self.act_first = act_first
        self.plain_last = plain_last

    def forward(self, x: Tensor) -> Tensor:
        # If `plain_last=True`, then `len(norms) = len(acts) = len(dropouts) = len(convs) - 1,
        # thus skipping the execution of the last layer inside the for-loop.
        for conv, norm, act, dropout in zip(self.convs, self.norms, self.acts, self.dropouts):
            x = conv(x)
            if self.act_first:
                x = act(x)
            x = norm(x)
            if not self.act_first:
                x = act(x)
            x = F.dropout(x, p=dropout, training=self.training)

        # If `plain_last=True`, then the last layer is executed here.
        if self.plain_last:
            x = self.convs[-1](x)

        return x

In [90]:
shared_mlp = SharedMLP([10, 20, 50, 100], dropout=None, plain_last=False)
shared_mlp

SharedMLP(
  (convs): ModuleList(
    (0): Conv1d(10, 20, kernel_size=(1,), stride=(1,))
    (1): Conv1d(20, 50, kernel_size=(1,), stride=(1,))
    (2): Conv1d(50, 100, kernel_size=(1,), stride=(1,))
  )
  (acts): ModuleList(
    (0-2): 3 x ReLU()
  )
  (norms): ModuleList(
    (0): BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)

In [92]:
x = torch.randn(32, 10, 32, 32)

x = torch.randn(32, 10, 32)
out = shared_mlp(x)

In [23]:
from typing import Any, Dict, List, Optional, Union

import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

from torch_pointcloud.layers._modules import MODULE_TYPE, REGISTERED_MODULE_TYPE, get_module
from torch_pointcloud.layers.activations import get_act
from torch_pointcloud.layers.norms import get_norm


class MLP(nn.Module):
    def __init__(
        self,
        dims: List[int],
        *,
        act: Optional[MODULE_TYPE] = "relu",
        act_first: bool = False,
        norm: Optional[MODULE_TYPE] = "batch_norm1d",
        dropout: float = 0.0,
        bias: Union[bool] = True,
        plain_last: bool = True,
    ) -> None:
        super().__init__()
        N = len(dims)
        N1 = N - 1
        N2 = N - 2 if plain_last else N1
        if N < 2:
            raise ValueError(f"The MLP must have at least 2 dimensions. Got {N}.")

        # Format activations (one for each layer except the last one)
        act = get_act(act) if act is not None else None
        # Format batch norms (one for each layer except the last one)
        norms = [get_norm(norm, dim) for dim in dims[1:]] if norm is not None else [nn.Identity()] * N2
        # Format dropouts (one for each layer except the last one)
        dropouts = [dropout] * N2
        # Format biases (one for each layer)
        biases = [bias] * N1

        # Sanity check
        if len(norms) != N2 or len(dropouts) != N2:
            raise ValueError(
                "The number of batch norm layers and dropouts must be equal to the number of layers (-1 if plain_last is `True`) in the MLP. "
                f"Got {len(norms)}, {len(dropouts)}, and {N1} (with {plain_last=}) respectively."
            )

        self.lins = nn.ModuleList(
            [nn.Linear(in_dim, out_dim, bias=bias) for in_dim, out_dim, bias in zip(dims[:-1], dims[1:], biases)]
        )
        self.norms = nn.ModuleList(norms)
        self.dropouts = dropouts
        self.act = act
        self.act_first = act_first
        self.plain_last = plain_last

    def forward(self, x: Tensor) -> Tensor:
        # If `plain_last=True`, then `len(norms) = len(acts) = len(dropouts) = len(linear_layers) - 1,
        # thus skipping the execution of the last layer inside the for-loop.
        for lin, norm, dropout in zip(self.lins, self.norms, self.dropouts):
            x = lin(x)
            if self.act and self.act_first:
                x = self.act(x)
            x = norm(x)
            if self.act and not self.act_first:
                x = self.act(x)
            x = F.dropout(x, p=dropout, training=self.training)

        # If `plain_last=True`, then the last layer is executed here.
        if self.plain_last:
            x = self.linear_layers[-1](x)

        return x

    def extra_repr(self) -> str:
        dims = [lin.in_features for lin in self.linear_layers] + [self.linear_layers[-1].out_features]
        return f"dims={dims}, act_first={self.act_first}, plain_last={self.plain_last}"


_convS: Dict[str, REGISTERED_MODULE_TYPE] = {
    "conv1d": nn.Conv1d,
    "conv2d": nn.Conv2d,
    "conv3d": nn.Conv3d,
}


def _get_conv(name: MODULE_TYPE, *args: Any, **kwargs: Any) -> nn.Module:
    return get_module(name, *args, registry=_convS, **kwargs)


class SharedMLP(nn.Module):
    def __init__(
        self,
        channels: List[int],
        *,
        act_first: bool = False,
        act: Optional[MODULE_TYPE] = "relu",
        norm: Optional[MODULE_TYPE] = "batch_norm1d",
        conv: MODULE_TYPE = "conv1d",
        dropout: float = 0.0,
        bias: Union[bool] = True,
        plain_last: bool = True,
    ):
        super().__init__()
        N = len(channels)
        N1 = N - 1
        N2 = N - 2 if plain_last else N1
        if N < 2:
            raise ValueError(f"The SharedMLP must have at least 2 channels. Got {N}.")

        # Format activations (one for each layer except the last one)
        act = get_act(act) if act is not None else None
        # Format batch norms (one for each layer except the last one)
        norms = [get_norm(norm, channel) for channel in channels[1:N2+1]] if norm is not None else [nn.Identity()] * N2
        # Format dropouts (one for each layer except the last one)
        dropouts = [dropout] * N2
        # Format biases (one for each layer)
        biases = [bias] * N1
        
        # Sanity check
        if len(norms) != N2 or len(dropouts) != N2 or len(biases) != N1:
            raise ValueError(
                "The number of batch norm layers, and dropouts must be equal to the number of layers (-1 if plain_last is `True`) in the SharedMLP. "
                f"Got {len(norms)=}, {len(dropouts)=}, and {len(channels)=} (with {plain_last=})."
            )

        convs = [
            _get_conv(conv, in_channels, out_channels, bias=bias, kernel_size=1, stride=1)
            for in_channels, out_channels, bias in zip(channels[:-1], channels[1:], biases)
        ]
        self.convs = nn.ModuleList(convs)
        self.norms = nn.ModuleList(norms)
        self.dropouts = dropouts
        self.act = act
        self.act_first = act_first
        self.plain_last = plain_last

    def forward(self, x: Tensor) -> Tensor:
        # If `plain_last=True`, then `len(norms) = len(dropouts) = len(convs) - 1,
        # thus skipping the execution of the last layer inside the for-loop.
        for conv, norm, dropout in zip(self.convs, self.norms, self.dropouts):
            x = conv(x)
            if self.act and self.act_first:
                x = self.act(x)
            x = norm(x)
            if self.act and not self.act_first:
                x = self.act(x)
            x = F.dropout(x, p=dropout, training=self.training)

        # If `plain_last=True`, then the last layer is executed here.
        if self.plain_last:
            x = self.convs[-1](x)

        return x

In [24]:
mlp = MLP([10, 20, 50, 100], dropout=0.2, norm=None)

In [28]:
shared_mlp = SharedMLP([10, 20, 50, 100], dropout=0.2, norm=None)

[0.2, 0.2] 2
